In [1]:
import pandas as pd
import numpy as np
import json
import yaml
from pathlib import Path
import sys
from IPython.display import display, HTML, Image, Markdown
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

sys.path.append('../')

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)    

## Cargado de los resultados de todos los experimentos

In [2]:
EXPERIMENTS_DIR = Path('../experiments')

all_metrics = []
experiment_artifacts = []
CLASS_NAMES = ['Normal', 'Robbery']

print(f"Buscando experimentos en {EXPERIMENTS_DIR.resolve()}")

for exp_dir in sorted(EXPERIMENTS_DIR.iterdir()):
    if not exp_dir.is_dir():
        continue

    exp_name = exp_dir.name

    metrics_path = exp_dir / 'results' / 'tables' / 'lstm_final_metrics.json'
    preds_path = exp_dir / 'results' / 'tables' / 'lstm_test_predictions.json'
    history_path = exp_dir / 'results' / 'tables' / 'lstm_training_history.json'
    plot_path = exp_dir / 'results' / 'plots' / '2_classification_analysis.png'
    config_path = exp_dir / 'config_run.yml'

    if metrics_path.exists() and preds_path.exists():
        print(f"Cargando datos de {exp_name}")
        try:
            with open(metrics_path, 'r') as f:
                final_metrics = json.load(f)

            with open(preds_path, 'r') as f:
                test_predictions = json.load(f)
            
            # Cargar balance mode del config
            balance_mode = 'none'
            if config_path.exists():
                with open(config_path, 'r') as f:
                    config = yaml.safe_load(f)
                    balance_mode = config.get('data_pipeline', {}).get('balance_mode', 'none')
            
            # Cargar y calcular datos de Overfitting
            ratio = np.nan
            risk = 'Desconocido'
            risk_score = 4
            
            if history_path.exists():
                with open(history_path, 'r') as f:
                    training_history = json.load(f)
                    
                best_epoch = final_metrics.get('best_epoch', None)
                train_losses = training_history.get('train_loss', [])
                val_losses = training_history.get('val_loss', [])
                
                if best_epoch is not None and isinstance(best_epoch, int):
                    train_losses = train_losses[:best_epoch]
                    val_losses = val_losses[:best_epoch]
                    
                if train_losses and val_losses:
                    avg_train_loss = np.mean(train_losses)
                    avg_val_loss = np.mean(val_losses)
                    ratio = avg_val_loss / (avg_train_loss + 1e-6)
                    
                    max_epochs_overfit = [max(0, val - train) for train, val in zip(train_losses, val_losses)]
                    avg_max_overfit = np.mean(max_epochs_overfit) if max_epochs_overfit else 0
                    
                    if ratio > 1.3 or avg_max_overfit > 0.15:
                        risk = 'Alto'
                        risk_score = 3
                    elif ratio > 1.1 or avg_max_overfit > 0.08:
                        risk = 'Moderado'
                        risk_score = 2
                    else:
                        risk = 'Bajo'
                        risk_score = 1

            y_true = test_predictions['labels']
            y_pred = test_predictions['preds']

            test_metrics_global = final_metrics.get('test_metrics', {})
            auc_score = test_metrics_global.get('auc', 0.0)

            report_dict = classification_report(y_true, y_pred, target_names=CLASS_NAMES, output_dict=True)

            robbery_metrics = report_dict.get('Robbery', {})
            robbery_recall = robbery_metrics.get('recall', 0.0)
            robbery_f1 = robbery_metrics.get('f1-score', 0.0)
            robbery_precision = robbery_metrics.get('precision', 0.0)

            all_metrics.append({
                'Experimento': exp_name,
                'AUC': auc_score,
                'F1': robbery_f1,
                'Recall': robbery_recall,
                'Precision': robbery_precision,
                'Accuracy': test_metrics_global.get('accuracy', 0.0),
                'Balance Mode': balance_mode,
                'Ratio (Val/Train)': ratio,
                'Riesgo Overfitting': risk,
                '_risk_score': risk_score
            })

            experiment_artifacts.append({
                'name': exp_name,
                'global_metrics': test_metrics_global,
                'classification_report': classification_report(y_true, y_pred, target_names=CLASS_NAMES),
                'plot_path': plot_path if plot_path.exists() else None
            })
        except Exception as e:
            print(f"Error al cargar {exp_name}: {e}")

Buscando experimentos en D:\Dataset\experiments
Cargando datos de exp_01
Cargando datos de exp_02
Cargando datos de exp_03
Cargando datos de exp_04
Cargando datos de exp_05
Cargando datos de exp_06
Cargando datos de exp_07
Cargando datos de exp_08
Cargando datos de exp_09
Cargando datos de exp_10
Cargando datos de exp_11
Cargando datos de exp_12
Cargando datos de exp_13
Cargando datos de exp_14
Cargando datos de exp_15
Cargando datos de exp_16
Cargando datos de exp_17
Cargando datos de exp_18
Cargando datos de exp_19
Cargando datos de exp_20
Cargando datos de exp_21
Cargando datos de exp_22
Cargando datos de exp_23
Cargando datos de exp_24
Cargando datos de exp_25
Cargando datos de exp_26
Cargando datos de exp_27
Cargando datos de exp_28
Cargando datos de exp_29
Cargando datos de exp_30
Cargando datos de exp_31
Cargando datos de exp_32
Cargando datos de exp_33
Cargando datos de exp_34
Cargando datos de exp_35
Cargando datos de exp_36
Cargando datos de exp_37
Cargando datos de exp_38
Ca

## Visualización de resultados individuales

In [3]:
'''
for artifact in experiment_artifacts:
    display(HTML(f"<h2>Resultados para: {artifact['name']}</h2>"))

    #métricas globales
    display(HTML("<h3>Métricas Globales de Test</h3>"))
    metrics_df = pd.DataFrame.from_dict(artifact['global_metrics'], orient='index', columns=['Valor'])
    metrics_df.index.name = 'Métrica'
    display(metrics_df.style.format({'Valor': '{:.4f}'}))

    #reporte de clasificación
    display(HTML("<h3>Reporte de Clasificación (Test)</h3>"))
    display(HTML(f"<pre>{artifact['classification_report']}</pre>"))

    #gráfico de análisis (matriz de confusión y Curva ROC)
    if artifact['plot_path']:
        display(HTML("<h3>Gráfico de Análisis</h3>"))
        display(Image(filename=artifact['plot_path'], width=900))
    
    display(HTML("<hr>"))
'''

'\nfor artifact in experiment_artifacts:\n    display(HTML(f"<h2>Resultados para: {artifact[\'name\']}</h2>"))\n\n    #métricas globales\n    display(HTML("<h3>Métricas Globales de Test</h3>"))\n    metrics_df = pd.DataFrame.from_dict(artifact[\'global_metrics\'], orient=\'index\', columns=[\'Valor\'])\n    metrics_df.index.name = \'Métrica\'\n    display(metrics_df.style.format({\'Valor\': \'{:.4f}\'}))\n\n    #reporte de clasificación\n    display(HTML("<h3>Reporte de Clasificación (Test)</h3>"))\n    display(HTML(f"<pre>{artifact[\'classification_report\']}</pre>"))\n\n    #gráfico de análisis (matriz de confusión y Curva ROC)\n    if artifact[\'plot_path\']:\n        display(HTML("<h3>Gráfico de Análisis</h3>"))\n        display(Image(filename=artifact[\'plot_path\'], width=900))\n\n    display(HTML("<hr>"))\n'

## Tabla Comparativa y Mejor Modelo

In [4]:
comparison_df = pd.DataFrame(all_metrics)
comparison_df = comparison_df.set_index('Experimento')

#criterio de ordenamiento
# mayor f1-score
# mayor recall para menos falsos positivos
# mayor auc
SORT_CRITERIA = ['Recall', 'F1', 'AUC']

comparison_df_sorted = comparison_df.sort_values(
    by=SORT_CRITERIA,
    ascending=[False, False, False]
)
comparison_df_sorted = comparison_df_sorted.reset_index()

display(HTML("<h2>Tabla Comparativa de Experimentos (Ordenada por Mejor Sensibilidad)</h2>"))
display(HTML("<p><b>Criterio de ordenamiento:</b> Máximo Recall, luego F1-Score, luego AUC.</p>"))

# Columnas a mostrar
base_columns = ['Experimento', 'AUC', 'F1', 'Recall', 'Precision', 'Accuracy', 'Balance Mode']
comparison_df_base = comparison_df_sorted[base_columns]

#tabla formateada
display(comparison_df_base.style.format({
    'Recall': '{:.4f}',
    'F1': '{:.4f}',
    'AUC': '{:.4f}',
    'Precision': '{:.4f}',
    'Accuracy': '{:.4f}'
}).background_gradient(cmap='Greens', subset=SORT_CRITERIA).set_properties(**{'text-align': 'center'}).hide(axis='index'))

Experimento,AUC,F1,Recall,Precision,Accuracy,Balance Mode
exp_68,0.8626,0.8235,0.8400,0.8077,0.8594,none
exp_39,0.8831,0.8077,0.8400,0.7778,0.8438,undersample
exp_69,0.8821,0.8077,0.8400,0.7778,0.8438,none
exp_75,0.8790,0.8077,0.8400,0.7778,0.8438,oversample
exp_44,0.9169,0.7925,0.8400,0.7500,0.8281,undersample
exp_41,0.8749,0.7636,0.8400,0.7000,0.7969,undersample
exp_22,0.8923,0.8163,0.8000,0.8333,0.8594,none
exp_45,0.8749,0.8163,0.8000,0.8333,0.8594,oversample
exp_24,0.8605,0.8000,0.8000,0.8000,0.8438,none
exp_43,0.8595,0.8000,0.8000,0.8000,0.8438,undersample


In [5]:
best_exp_name = comparison_df_sorted.index[0]
best_recall = comparison_df_sorted.iloc[0]['Recall']
best_f1 = comparison_df_sorted.iloc[0]['F1']
best_auc = comparison_df_sorted.iloc[0]['AUC']

display(Markdown(f"""
## Conclusión del mejor experimento

Basado en el criterio de priorizar la reducción de falsos negativos (mayor recall) sin  sacrificar la media armónica entre recall y precisión (F1-Score), ignorando la posibilidad de sobre ajuste, el mejor modelo es:

**`{best_exp_name}`**

Este experimento logró:
* **Recall:** {best_recall:.4f}
* **F1-Score:** {best_f1:.4f}
* **AUC:** {best_auc:.4f}
"""))


## Conclusión del mejor experimento

Basado en el criterio de priorizar la reducción de falsos negativos (mayor recall) sin  sacrificar la media armónica entre recall y precisión (F1-Score), ignorando la posibilidad de sobre ajuste, el mejor modelo es:

**`0`**

Este experimento logró:
* **Recall:** 0.8400
* **F1-Score:** 0.8235
* **AUC:** 0.8626


## Tabla Comparativa por Riesgo de Overfitting

In [6]:
def color_risk(val):
    if val == 'Alto':
        return 'background-color: #ffcccc; color: black;'  # Rojo claro
    elif val == 'Moderado':
        return 'background-color: #ffffcc; color: black;'  # Amarillo claro
    elif val == 'Bajo':
        return 'background-color: #ccffcc; color: black;'  # Verde claro
    return ''

overfit_sorted_df = comparison_df.sort_values(
    by=['_risk_score', 'Ratio (Val/Train)'], 
    ascending=[True, True]
)
overfit_sorted_df = overfit_sorted_df.reset_index()

display(HTML("<h2>Experimentos Ordenados por Riesgo de Overfitting (Menor a Mayor)</h2>"))

columns_to_show = ['Experimento', 'Ratio (Val/Train)', 'Riesgo Overfitting', 'Recall', 'F1', 'AUC']

display(overfit_sorted_df[columns_to_show].style.format({
    'Recall': '{:.4f}',
    'F1': '{:.4f}',
    'AUC': '{:.4f}',
    'Ratio (Val/Train)': '{:.4f}'
}).map(color_risk, subset=['Riesgo Overfitting']).set_properties(**{'text-align': 'center'}).hide(axis='index'))

Experimento,Ratio (Val/Train),Riesgo Overfitting,Recall,F1,AUC
exp_69,0.8708,Bajo,0.8400,0.8077,0.8821
exp_77,0.8748,Bajo,0.7600,0.7917,0.8585
exp_49,0.9579,Bajo,0.7200,0.7347,0.7856
exp_51,0.9716,Bajo,0.6000,0.6818,0.8349
exp_60,0.9870,Bajo,0.6800,0.7234,0.8205
exp_74,0.9894,Bajo,0.6400,0.6667,0.8051
exp_57,1.0001,Bajo,0.6000,0.7143,0.8174
exp_32,1.0021,Bajo,0.3200,0.4848,0.8010
exp_30,1.0101,Bajo,0.4800,0.6000,0.8072
exp_72,1.0315,Bajo,0.7200,0.7200,0.8451


## Interpretación de Métricas de Sobreajuste

### Ratio (Val Loss / Train Loss)
- **Ratio ≈ 1.0**: Perfect fit (sin sobreajuste esperado)
- **Ratio 1.0 - 1.1**: Buena generalización, poco sobreajuste
- **Ratio 1.1 - 1.3**: Sobreajuste moderado
- **Ratio > 1.3**: Sobreajuste severo

### Clasificación de Riesgo
- **Bajo**: Modeló generaliza bien, se espera buen rendimiento en datos nuevos
- **Moderado**: Cierto sobreajuste, considerar regularización
- **Alto**: Sobreajuste severo, modelo probablemente no generalizará bien

## Ranking Global (Mayor Desempeño + Menor Sobreajuste)
En esta sección se combinan ambas perspectivas. Se priorizan los modelos que tienen el menor riesgo de Sobreajuste y que, a su vez, maximizan las métricas de desempeño para la clase minoritaria (Recall, luego F1-score y AUC).

In [7]:
# Criterios combinados: Menor riesgo de overfitting primero, luego mayor Recall, F1 y AUC
SORT_GLOBAL_CRITERIA = ['_risk_score', 'Recall', 'F1', 'AUC']

global_sorted_df = comparison_df.sort_values(
    by=SORT_GLOBAL_CRITERIA,
    ascending=[True, False, False, False]
)
global_sorted_df = global_sorted_df.reset_index()

display(HTML("<h2>Ranking Global de Experimentos</h2>"))

columns_global = ['Experimento', 'Recall', 'F1', 'AUC', 'Precision', 'Accuracy', 'Ratio (Val/Train)', 'Riesgo Overfitting']

display(global_sorted_df[columns_global].style.format({
    'Recall': '{:.4f}',
    'F1': '{:.4f}',
    'AUC': '{:.4f}',
    'Precision': '{:.4f}',
    'Accuracy': '{:.4f}',
    'Ratio (Val/Train)': '{:.4f}'
}).map(color_risk, subset=['Riesgo Overfitting']).background_gradient(cmap='Greens', subset=['Recall', 'F1', 'AUC']).set_properties(**{'text-align': 'center'}).hide(axis='index'))

# Conclusión del mejor modelo global
if not global_sorted_df.empty:
    best_global_exp_name = global_sorted_df.iloc[0]['Experimento']
    best_g_recall = global_sorted_df.iloc[0]['Recall']
    best_g_f1 = global_sorted_df.iloc[0]['F1']
    best_g_auc = global_sorted_df.iloc[0]['AUC']
    best_g_risk = global_sorted_df.iloc[0]['Riesgo Overfitting']
    
    display(Markdown(f"""
### Conclusión del Mejor Modelo Global

Al realizar el balance entre el mejor desempeño predictivo priorizando la reducción de falsos negativos (Recall), sin sacrificar la media armónica y considerando la capacidad del modelo para generalizar sin incurrir en alto sobreajuste, el experimento con mejor desempeño es:

**`{best_global_exp_name}`**

Este experimento cuenta con los siguientes resultados:
* **Riesgo Overfitting:** {best_g_risk}
* **Recall:** {best_g_recall:.4f}
* **F1-Score:** {best_g_f1:.4f}
* **AUC:** {best_g_auc:.4f}
"""))

Experimento,Recall,F1,AUC,Precision,Accuracy,Ratio (Val/Train),Riesgo Overfitting
exp_69,0.8400,0.8077,0.8821,0.7778,0.8438,0.8708,Bajo
exp_77,0.7600,0.7917,0.8585,0.8261,0.8438,0.8748,Bajo
exp_71,0.7200,0.7347,0.8667,0.7500,0.7969,1.0991,Bajo
exp_49,0.7200,0.7347,0.7856,0.7500,0.7969,0.9579,Bajo
exp_72,0.7200,0.7200,0.8451,0.7200,0.7812,1.0315,Bajo
exp_82,0.6800,0.7727,0.8338,0.8947,0.8438,1.0579,Bajo
exp_25,0.6800,0.7556,0.8277,0.8500,0.8281,1.0350,Bajo
exp_60,0.6800,0.7234,0.8205,0.7727,0.7969,0.9870,Bajo
exp_59,0.6400,0.7273,0.8738,0.8421,0.8125,1.0621,Bajo
exp_46,0.6400,0.6809,0.8646,0.7273,0.7656,1.0991,Bajo



### Conclusión del Mejor Modelo Global

Al realizar el balance entre el mejor desempeño predictivo priorizando la reducción de falsos negativos (Recall), sin sacrificar la media armónica y considerando la capacidad del modelo para generalizar sin incurrir en alto sobreajuste, el experimento con mejor desempeño es:

**`exp_69`**

Este experimento cuenta con los siguientes resultados:
* **Riesgo Overfitting:** Bajo
* **Recall:** 0.8400
* **F1-Score:** 0.8077
* **AUC:** 0.8821


## Exportar Resultados a HTML

In [8]:
import os
from datetime import datetime

export_dir = Path('../results/experiments')
export_dir.mkdir(parents=True, exist_ok=True)

export_path = export_dir / 'compare_experiments_report.html'

# Extraer el HTML de los objetos Styler con el formato de las celdas
html_table1 = comparison_df_base.style.format({
    'Recall': '{:.4f}',
    'F1': '{:.4f}',
    'AUC': '{:.4f}',
    'Precision': '{:.4f}',
    'Accuracy': '{:.4f}'
}).background_gradient(cmap='Greens', subset=SORT_CRITERIA).set_properties(**{'text-align': 'center'}).hide(axis='index').to_html()

html_table2 = overfit_sorted_df[columns_to_show].style.format({
    'Recall': '{:.4f}',
    'F1': '{:.4f}',
    'AUC': '{:.4f}',
    'Ratio (Val/Train)': '{:.4f}'
}).map(color_risk, subset=['Riesgo Overfitting']).set_properties(**{'text-align': 'center'}).hide(axis='index').to_html()

html_table3 = global_sorted_df[columns_global].style.format({
    'Recall': '{:.4f}',
    'F1': '{:.4f}',
    'AUC': '{:.4f}',
    'Precision': '{:.4f}',
    'Accuracy': '{:.4f}',
    'Ratio (Val/Train)': '{:.4f}'
}).map(color_risk, subset=['Riesgo Overfitting']).background_gradient(cmap='Greens', subset=['Recall', 'F1', 'AUC']).set_properties(**{'text-align': 'center'}).hide(axis='index').to_html()

# Limpiar las clases por defecto de pandas y reemplazar por las de la tabla custom
def add_table_styles(html_str):
    return html_str.replace('<table id="T_', '<table class="dataframe" id="T_')

html_table1 = add_table_styles(html_table1)
html_table2 = add_table_styles(html_table2)
html_table3 = add_table_styles(html_table3)

html_content = f"""<!DOCTYPE html>
<html lang='es'>
<head>
  <meta charset='utf-8' />
  <meta name='viewport' content='width=device-width, initial-scale=1' />
  <title>Reporte Comparativo - Experimentos</title>
  <style>
    :root {{
      --bg: #f7f8fa;
      --card: #ffffff;
      --ink: #1f2937;
      --muted: #6b7280;
      --brand: #1d4ed8;
      --line: #e5e7eb;
      --hero-bg: linear-gradient(135deg, #dbe9f4 0%, #cbd5e1 100%);
      --hero-text: #000000;
      --th-bg: #f3f4f6;
    }}
    
    html.dark-mode {{
      --bg: #1a1a1a;
      --card: #2a2a2a;
      --ink: #f0f0f0;
      --muted: #a0a0a0;
      --brand: #4a9eff;
      --line: #404040;
      --hero-bg: linear-gradient(135deg, #334155 0%, #1e293b 100%);
      --hero-text: #ffffff;
      --th-bg: #333333;
    }}
    
    body {{
      margin: 0;
      font-family: 'Segoe UI', Tahoma, sans-serif;
      background: var(--bg);
      color: var(--ink);
      transition: background-color 0.3s, color 0.3s;
    }}
    
    .container {{
      max-width: 1400px;
      margin: 0 auto;
      padding: 24px;
    }}
    
    .theme-toggle {{
      position: fixed;
      top: 20px;
      right: 20px;
      z-index: 1000;
      display: flex;
      align-items: center;
      gap: 10px;
    }}
    
    .toggle-label {{
      font-size: 14px;
      color: var(--muted);
    }}
    
    .toggle-switch {{
      position: relative;
      width: 60px;
      height: 30px;
      background: var(--line);
      border-radius: 15px;
      cursor: pointer;
      transition: background 0.3s;
      border: 2px solid var(--line);
    }}
    
    .toggle-switch.active {{
      background: #4a9eff;
      border-color: #4a9eff;
    }}
    
    .toggle-slider {{
      position: absolute;
      top: 2px;
      left: 2px;
      width: 24px;
      height: 24px;
      background: white;
      border-radius: 50%;
      transition: left 0.3s;
      display: flex;
      align-items: center;
      justify-content: center;
      font-size: 14px;
    }}
    
    .toggle-switch.active .toggle-slider {{
      left: 32px;
    }}
    
    .toggle-slider {{
      background-image: url("data:image/svg+xml;utf8,<svg xmlns='http://www.w3.org/2000/svg' viewBox='0 0 24 24' fill='none' stroke='%23FFB81C' stroke-width='2' stroke-linecap='round' stroke-linejoin='round'><circle cx='12' cy='12' r='5'/><line x1='12' y1='1' x2='12' y2='3'/><line x1='12' y1='21' x2='12' y2='23'/><line x1='4.22' y1='4.22' x2='5.64' y2='5.64'/><line x1='18.36' y1='18.36' x2='19.78' y2='19.78'/><line x1='1' y1='12' x2='3' y2='12'/><line x1='21' y1='12' x2='23' y2='12'/><line x1='4.22' y1='19.78' x2='5.64' y2='18.36'/><line x1='18.36' y1='5.64' x2='19.78' y2='4.22'/></svg>");
      background-size: 14px 14px;
      background-repeat: no-repeat;
      background-position: center;
    }}
    
    .toggle-switch.active .toggle-slider {{
      background-image: url("data:image/svg+xml;utf8,<svg xmlns='http://www.w3.org/2000/svg' viewBox='0 0 24 24' fill='%234B7BE5'><path d='M21 12.79A9 9 0 1 1 11.21 3 7 7 0 0 0 21 12.79z'/></svg>");
      background-size: 14px 14px;
      background-repeat: no-repeat;
      background-position: center;
    }}
    
    .hero {{
      background: var(--hero-bg);
      color: var(--hero-text);
      border-radius: 14px;
      padding: 24px;
      box-shadow: 0 10px 30px rgba(0,0,0,0.15);
      margin-bottom: 20px;
    }}
    
    .hero h1 {{ margin: 0 0 10px 0; font-size: 28px; }}
    .hero p {{ margin: 2px 0; opacity: 0.95; }}
    
    .card {{
      background: var(--card);
      border: 1px solid var(--line);
      border-radius: 12px;
      padding: 24px;
      margin-bottom: 24px;
      transition: border-color 0.3s, background 0.3s;
    }}
    
    .card h2 {{ margin: 0 0 10px 0; font-size: 18px; color: var(--brand); }}
    .card p {{ margin: 0 0 20px 0; font-size: 15px; color: var(--muted); }}
    
    /* Configuración de tablas base en modo oscuro */
    table {{ width: 100%; border-collapse: collapse; font-size: 14px; overflow-x: auto; text-align: center; margin-bottom: 16px; border: 1px solid var(--line); }}
    th, td {{ border-bottom: 1px solid var(--line); padding: 12px; font-weight: normal; }}
    th {{ background: var(--th-bg); font-weight: 600; text-align: center; color: var(--ink); border-right: 1px solid var(--line); }}
    td {{ color: var(--ink); border-right: 1px solid var(--line); text-align: center; }}
    
    /* Manejo del color del dataframe vs modo oscuro para no esconder las letras y adaptarlas a fondo fuerte */
    html.dark-mode td[style*="background-color: #ffcccc"] {{ background-color: #7b1fa2 !important; color: white !important; font-weight: bold; border-color: #404040; }}
    html.dark-mode td[style*="background-color: #ffffcc"] {{ background-color: #e65100 !important; color: white !important; font-weight: bold; border-color: #404040; }}
    html.dark-mode td[style*="background-color: #ccffcc"] {{ background-color: #1b5e20 !important; color: white !important; font-weight: bold; border-color: #404040; }}
    
    .table-wrapper {{ overflow-x: auto; border-radius: 8px; border: 1px solid var(--line); background-color: var(--card); }}
    
    .section-divider {{ border-top: 2px solid var(--ink); margin: 30px 0 20px 0; opacity: 0.2; }}
    
    .conclusion-box {{
      background: var(--card);
      border-left: 6px solid #4a9eff;
      border-radius: 12px;
      padding: 24px 30px;
      margin-top: 30px;
      box-shadow: 0 4px 15px rgba(0,0,0,0.05);
      border-top: 1px solid var(--line);
      border-right: 1px solid var(--line);
      border-bottom: 1px solid var(--line);
    }}
    
    .conclusion-box h3 {{ margin-top: 0; color: #4a9eff; font-size: 22px; margin-bottom: 15px; }}
    .conclusion-box p {{ font-size: 16px; color: var(--ink); line-height: 1.6; margin-bottom: 15px; }}
    
    ul {{ padding-left: 25px; margin: 0; }}
    li {{ margin-bottom: 10px; font-size: 15px; color: var(--ink); }}
    li strong {{ color: var(--brand); }}
    
    .foot {{ color: var(--muted); font-size: 12px; margin-top: 30px; text-align: center; }}
  </style>
</head>
<body>
  <div class='theme-toggle'>
    <span class='toggle-label'>Tema</span>
    <div class='toggle-switch' id='themeToggle'>
      <div class='toggle-slider'></div>
    </div>
  </div>
  
  <div class='container'>
    <section class='hero'>
      <h1>Reporte Comparativo - Análisis de Experimentos</h1>
      <p><strong>Total Analizados:</strong> {len(comparison_df)} experimentos</p>
      <p><strong>Generado:</strong> {datetime.now().strftime('%d/%m/%Y %H:%M:%S')}</p>
    </section>

    <section class='card'>
      <h2>1. Tabla de Rendimiento (Ordenada por Sensibilidad)</h2>
      <p>Resultados base excluyendo información de sobreajuste, priorizando minimizar los falsos negativos (<strong>Recall</strong>, <strong>F1-Score</strong> y <strong>AUC</strong>).</p>
      <div class='table-wrapper'>
        {html_table1}
      </div>
    </section>

    <div class='section-divider'></div>

    <section class='card'>
      <h2>2. Tabla Analítica de Overfitting (Mayor Generalización)</h2>
      <p>Aísla la capacidad de generalización calculando el ratio de la pérdida de validación respecto a la del entrenamiento.</p>
      <div class='table-wrapper'>
        {html_table2}
      </div>
    </section>

    <div class='section-divider'></div>

    <section class='card'>
      <h2>3. Ranking Global Analítico (Menor Overfitting + Mayor Rendimiento)</h2>
      <p>Tabla definitiva que prioriza un riesgo de validación estable combinado a la capacidad predictiva.</p>
      <div class='table-wrapper'>
        {html_table3}
      </div>
    </section>

    <div class='conclusion-box'>
        <h3>Conclusión del Mejor Modelo</h3>
        <p>Al realizar el balance entre el mejor desempeño predictivo priorizando la reducción de falsos negativos (Recall), sin sacrificar la media armónica y considerando la capacidad del modelo para generalizar sin incurrir en alto sobreajuste, el mejor experimento es:</p>
        <p style="font-size: 1.5em; font-weight: bold; margin: 15px 0;">{best_global_exp_name}</p>
        <ul>
            <li><strong>Riesgo Overfitting:</strong> {best_g_risk}</li>
            <li><strong>Recall:</strong> {best_g_recall:.4f}</li>
            <li><strong>F1-Score:</strong> {best_g_f1:.4f}</li>
            <li><strong>AUC:</strong> {best_g_auc:.4f}</li>
        </ul>
    </div>
    
    <p class='foot'>Reporte generado automáticamente desde notebooks/4_compare_experiments.ipynb.</p>
  </div>
  
  <script>
    const themeToggle = document.getElementById('themeToggle');
    const html = document.documentElement;
    
    // Check local storage or system preference
    const savedTheme = localStorage.getItem('theme') || 'light';
    if (savedTheme === 'dark') {{
      html.classList.add('dark-mode');
      themeToggle.classList.add('active');
    }}
    
    themeToggle.addEventListener('click', () => {{
      html.classList.toggle('dark-mode');
      themeToggle.classList.toggle('active');
      
      const isDark = html.classList.contains('dark-mode');
      localStorage.setItem('theme', isDark ? 'dark' : 'light');
    }});
  </script>
</body>
</html>
"""

# Escribir HTML al archivo
with open(export_path, 'w', encoding='utf-8') as f:
    f.write(html_content)

print(f"Reporte guardado exitosamente en: {export_path.resolve()}")

Reporte guardado exitosamente en: D:\Dataset\results\experiments\compare_experiments_report.html
